## Setup

In [1]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [2]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [3]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [5]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='tr')

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [6]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-ORG', 'B-LOC', 'I-PER', 'I-ORG', 'I-LOC', 'B-PER', 'O'}


# Evaluate model

In [7]:
alignment = {
'B-LOC': 'B-LOC',
'B-MISC': 'O',
'B-ORG': 'B-ORG',
'I-LOC': 'I-LOC',
'I-MISC': 'O',
'I-ORG': 'I-ORG',
'I-PER': 'I-PER',
'O': 'O'
}

model_name = "xlm-roberta-large-finetuned-conll03-english"
model_name_output = 'xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at xlm-roberta-large-finetuned-conll03-english were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# model_evaluation.model.config.id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'I-LOC',
 4: 'I-MISC',
 5: 'I-ORG',
 6: 'I-PER',
 7: 'O'}

### wikiann

In [9]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [10]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6513,0.7092,0.6790,4914
1,ORG,0.6950,0.4466,0.5437,4154
2,PER,0.7995,0.8728,0.8345,4519
3,micro,0.7167,0.6833,0.6996,13587
4,macro,0.7153,0.6762,0.6858,13587
5,weighted,0.7140,0.6833,0.6894,13587


In [11]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0000,0.0000,0.0000,4914
1,B-ORG,0.0000,0.0000,0.0000,4154
2,B-PER,0.0000,0.0000,0.0000,4519
3,I-LOC,0.2091,0.4855,0.2923,3005
4,I-ORG,0.4806,0.3820,0.4257,6979
5,I-PER,0.4932,0.8613,0.6272,5694
6,O,0.8564,0.9817,0.9148,46466
7,accuracy,0.7215,75731,None,None
8,macro,0.2913,0.3872,0.3228,75731
9,weighted,0.6151,0.7215,0.6593,75731
